In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.order_items"
)

silver_df = spark.table(
    "customer360_dev.silver.order_items"
)

reject_df = spark.table(
    "customer360_dev.quarantine.order_item_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 15,000,186
Silver rows   : 11,400,000
Rejected rows : 3,600,186


In [0]:
orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

products_df = (
    spark.table("customer360_dev.silver.products")
    .select("product_id")
)

validated_df = (
    bronze_df
    .withColumn(
        "quantity_cast",
        F.col("quantity").cast("int")
    )
    .withColumn(
        "unit_price_cast",
        F.col("unit_price").cast("decimal(14,2)")
    )
    .withColumn(
        "discount_amount_cast",
        F.col("discount_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "tax_amount_cast",
        F.col("tax_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "line_amount_cast",
        F.col("line_amount").cast("decimal(14,2)")
    )
    .withColumn(
        "item_status_clean",
        F.upper(F.trim("item_status"))
    )
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
    .join(
        products_df.withColumn(
            "_product_exists",
            F.lit(True)
        ),
        on="product_id",
        how="left"
    )
)

In [0]:
display(validated_df.limit(10))

product_id,order_id,order_item_id,quantity,unit_price,discount_amount,tax_amount,line_amount,item_status,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,quantity_cast,unit_price_cast,discount_amount_cast,tax_amount_cast,line_amount_cast,item_status_clean,_order_exists,_product_exists
PROD000026474,ORD0001000854,ITEM00006000854,5,8969.38,10.00,2241.85,47078.75,CANCELLED,2026-05-10 02:34:02,2026-05-23 21:57:51,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,8969.38,10.00,2241.85,47078.75,CANCELLED,true,true
PROD000050344,ORD0001001624,ITEM00006001624,5,571.28,0.00,142.82,2999.22,CANCELLED,2025-09-23 16:44:52,2025-10-06 11:07:41,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,571.28,0.00,142.82,2999.22,CANCELLED,true,true
PROD000017794,ORD0001000574,ITEM00006000574,5,10777.78,10.00,2693.95,56572.85,CANCELLED,2025-08-03 10:14:22,2025-08-16 05:37:11,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,10777.78,10.00,2693.95,56572.85,CANCELLED,true,true
PROD000020646,ORD0001000666,ITEM00006000666,2,985.02,10.00,98.00,2058.04,DELIVERED,2025-11-03 06:06:18,2025-11-16 01:29:07,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,2,985.02,10.00,98.00,2058.04,DELIVERED,true,true
PROD000022444,ORD0001000724,ITEM00006000724,5,3448.28,0.00,862.07,18103.47,CANCELLED,2025-12-31 04:44:52,2026-01-13 23:07:41,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,3448.28,0.00,862.07,18103.47,CANCELLED,true,true
PROD000030814,ORD0001000994,ITEM00006000994,5,1215.18,10.00,303.30,6369.20,CANCELLED,2024-01-02 22:14:22,2024-01-15 17:37:11,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,1215.18,10.00,303.30,6369.20,CANCELLED,true,true
PROD000031868,ORD0001001028,ITEM00006001028,4,2659.16,0.00,531.83,11168.47,SHIPPED,2024-02-05 20:28:44,2024-02-18 15:51:33,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,4,2659.16,0.00,531.83,11168.47,SHIPPED,true,true
PROD000046872,ORD0001001512,ITEM00006001512,3,9514.64,0.00,1427.20,29971.12,RETURNED,2025-06-03 00:12:36,2025-06-16 19:35:25,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,3,9514.64,0.00,1427.20,29971.12,RETURNED,true,true
PROD000008308,ORD0001000268,ITEM00006000268,4,11481.96,0.00,2296.39,48224.23,SHIPPED,2024-10-01 04:08:04,2024-10-14 23:31:53,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,4,11481.96,0.00,2296.39,48224.23,SHIPPED,true,true
PROD000035092,ORD0001001132,ITEM00006001132,3,7076.04,0.00,1061.41,22289.53,RETURNED,2024-05-19 04:32:16,2024-06-01 23:55:05,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,3,7076.04,0.00,1061.41,22289.53,RETURNED,true,true


In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_order_item_id",
        F.col("order_item_id").isNull()
        | (F.trim(F.col("order_item_id")) == "")
    )
    .withColumn(
        "dq_missing_order_id",
        F.col("order_id").isNull()
        | (F.trim(F.col("order_id")) == "")
    )
    .withColumn(
        "dq_missing_product_id",
        F.col("product_id").isNull()
        | (F.trim(F.col("product_id")) == "")
    )
    .withColumn(
        "dq_order_not_found",
        F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_product_not_found",
        F.col("_product_exists").isNull()
    )
    .withColumn(
        "dq_invalid_quantity",
        F.col("quantity_cast").isNull()
        | (F.col("quantity_cast") <= 0)
    )
    .withColumn(
        "dq_invalid_unit_price",
        F.col("unit_price_cast").isNull()
        | (F.col("unit_price_cast") < 0)
    )
    .withColumn(
        "dq_invalid_discount_amount",
        F.col("discount_amount_cast").isNull()
        | (F.col("discount_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_tax_amount",
        F.col("tax_amount_cast").isNull()
        | (F.col("tax_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_line_amount",
        F.col("line_amount_cast").isNull()
        | (F.col("line_amount_cast") < 0)
    )
    .withColumn(
        "dq_invalid_item_status",
        ~F.col("item_status_clean").isin(
            "CREATED",
            "CONFIRMED",
            "SHIPPED",
            "DELIVERED",
            "CANCELLED",
            "RETURNED"
        )
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "calculated_line_amount",
        (
            (
                F.col("quantity_cast")
                * F.col("unit_price_cast")
            )
            - F.col("discount_amount_cast")
            + F.col("tax_amount_cast")
        ).cast("decimal(14,2)")
    )
    .withColumn(
        "dq_line_amount_mismatch",
        F.abs(
            F.col("calculated_line_amount")
            - F.col("line_amount_cast")
        ) > F.lit(0.01)
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_order_item_id")
    | F.col("dq_missing_order_id")
    | F.col("dq_missing_product_id")
    | F.col("dq_order_not_found")
    | F.col("dq_product_not_found")
    | F.col("dq_invalid_quantity")
    | F.col("dq_invalid_unit_price")
    | F.col("dq_invalid_discount_amount")
    | F.col("dq_invalid_tax_amount")
    | F.col("dq_invalid_line_amount")
    | F.col("dq_invalid_item_status")
    | F.col("dq_line_amount_mismatch")
)

In [0]:
display(validated_df.limit(10))

product_id,order_id,order_item_id,quantity,unit_price,discount_amount,tax_amount,line_amount,item_status,created_at,updated_at,_ingestion_timestamp,_source_file,_source_system,_load_type,quantity_cast,unit_price_cast,discount_amount_cast,tax_amount_cast,line_amount_cast,item_status_clean,_order_exists,_product_exists,dq_missing_order_item_id,dq_missing_order_id,dq_missing_product_id,dq_order_not_found,dq_product_not_found,dq_invalid_quantity,dq_invalid_unit_price,dq_invalid_discount_amount,dq_invalid_tax_amount,dq_invalid_line_amount,dq_invalid_item_status,calculated_line_amount,dq_line_amount_mismatch,dq_critical_failure
PROD000026474,ORD0001000854,ITEM00006000854,5,8969.38,10.00,2241.85,47078.75,CANCELLED,2026-05-10 02:34:02,2026-05-23 21:57:51,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,8969.38,10.00,2241.85,47078.75,CANCELLED,true,true,false,false,false,false,false,false,false,false,false,false,false,47078.75,false,false
PROD000050344,ORD0001001624,ITEM00006001624,5,571.28,0.00,142.82,2999.22,CANCELLED,2025-09-23 16:44:52,2025-10-06 11:07:41,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,571.28,0.00,142.82,2999.22,CANCELLED,true,true,false,false,false,false,false,false,false,false,false,false,false,2999.22,false,false
PROD000017794,ORD0001000574,ITEM00006000574,5,10777.78,10.00,2693.95,56572.85,CANCELLED,2025-08-03 10:14:22,2025-08-16 05:37:11,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,10777.78,10.00,2693.95,56572.85,CANCELLED,true,true,false,false,false,false,false,false,false,false,false,false,false,56572.85,false,false
PROD000020646,ORD0001000666,ITEM00006000666,2,985.02,10.00,98.00,2058.04,DELIVERED,2025-11-03 06:06:18,2025-11-16 01:29:07,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,2,985.02,10.00,98.00,2058.04,DELIVERED,true,true,false,false,false,false,false,false,false,false,false,false,false,2058.04,false,false
PROD000022444,ORD0001000724,ITEM00006000724,5,3448.28,0.00,862.07,18103.47,CANCELLED,2025-12-31 04:44:52,2026-01-13 23:07:41,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,3448.28,0.00,862.07,18103.47,CANCELLED,true,true,false,false,false,false,false,false,false,false,false,false,false,18103.47,false,false
PROD000030814,ORD0001000994,ITEM00006000994,5,1215.18,10.00,303.30,6369.20,CANCELLED,2024-01-02 22:14:22,2024-01-15 17:37:11,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,5,1215.18,10.00,303.30,6369.20,CANCELLED,true,true,false,false,false,false,false,false,false,false,false,false,false,6369.20,false,false
PROD000031868,ORD0001001028,ITEM00006001028,4,2659.16,0.00,531.83,11168.47,SHIPPED,2024-02-05 20:28:44,2024-02-18 15:51:33,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,4,2659.16,0.00,531.83,11168.47,SHIPPED,true,true,false,false,false,false,false,false,false,false,false,false,false,11168.47,false,false
PROD000046872,ORD0001001512,ITEM00006001512,3,9514.64,0.00,1427.20,29971.12,RETURNED,2025-06-03 00:12:36,2025-06-16 19:35:25,2026-09-26T10:00:02.547Z,dbfs:/Volumes/customer360_dev/staging/source_files/order_items/initial/order_items_initial_013.csv,ORDER_MANAGEMENT,INITIAL,3,9514.64,0.00,1427.20,29971.12,RETURNED,true,true,false,false,false,false,false,false,false,false,false,false,false,29971.12,false,false
PROD000008308,ORD0001000268,ITEM00006000268,4,11481.96,0.00,2296.39,48224.23,SH

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Valid candidates before dedup: 11,400,000
Duplicate rows removed: 0


In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 15,000,186
Silver rows        : 11,400,000
Rejected rows      : 3,600,186
Duplicates removed : 0
Accounted for      : 15,000,186
Difference         : 0


In [0]:
result = [
    {
        "entity": "order_items",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
15000186,15000186,0,0,order_items,3600186,11400000,PASS


In [0]:
orphan_orders = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.orders"
        ).select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

print(
    "Order Items with missing Silver Order:",
    orphan_orders
)

Order Items with missing Silver Order: 0


In [0]:
orphan_products = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.products"
        ).select("product_id"),
        on="product_id",
        how="left_anti"
    )
    .count()
)

print(
    "Order Items with missing Silver Product:",
    orphan_products
)

Order Items with missing Silver Product: 0


In [0]:
warning_count = (
    silver_df
    .filter(
        F.col(
            "dq_warning_discount_exceeds_line_gross"
        ) == True
    )
    .count()
)

print(
    "Discount exceeds line gross warnings:",
    warning_count
)

Discount exceeds line gross warnings: 0
